# ЛР 02.2 — Локальный разбор ошибок и сегментов (TODO)

## Цель
- взять лучшую пару `model + feature_set` из ЛР 01;
- разобрать наиболее уверенные false positive и false negative;
- построить локальные объяснения ошибок;
- дополнить локальный анализ сегментной сводкой по ошибкам.

In [1]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '02-model-interpretability-and-explainability',
    cwd.parent / '02-model-interpretability-and-explainability',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 02 или из корня репозитория.')
spec = importlib.util.spec_from_file_location('lab02_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

## Шаг 1. Выбор лучшей пары `model + feature_set` из ЛР 01

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [2]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

model_factories = {
    'LogisticRegression': lambda: LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
    ),
    'LinearSVC': lambda: LinearSVC(C=1.0, class_weight='balanced', random_state=SEED),
}

prepared = {}
selection_rows = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_test, y_train, y_test = lab.train_test_split_stratified(x, y)
    best_config = lab.choose_best_model_config(model_results, feature_sets, dataset_name)
    prepared[dataset_name] = {
        'x_train': x_train,
        'x_test': x_test,
        'y_train': y_train,
        'y_test': y_test,
        'best_config': best_config,
        'selected_features': feature_sets[dataset_name][best_config['feature_set']],
    }
    selection_rows.append(best_config)

selection_summary = pd.DataFrame(selection_rows)
selection_summary

,dataset,feature_set,model,roc_auc,f1,accuracy
0,medical,set_A_wrapper,LinearSVC,0.760502,0.490909,0.688889
1,finance,set_B_tree,LinearSVC,0.716154,0.559524,0.663636


## Шаг 2. Обучение выбранных моделей и локальные объяснения ошибок

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [3]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

artifacts = {}
error_frames = []
segment_frames = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, ctx in prepared.items():
    best_config = ctx['best_config']
    model_name = best_config['model']
    artifact = lab.fit_selected_model(
        ctx['x_train'],
        ctx['x_test'],
        ctx['y_train'],
        ctx['y_test'],
        ctx['selected_features'],
        model_factories[model_name](),
    )
    artifacts[dataset_name] = artifact

    error_frames.append(
        lab.build_error_case_explanations(
            artifact=artifact,
            dataset_name=dataset_name,
            model_name=model_name,
            feature_set_name=best_config['feature_set'],
        )
    )
    segment_frames.append(lab.build_default_segment_tables(artifact, dataset_name))

error_case_explanations = pd.concat(error_frames, ignore_index=True)
segment_error_summary = pd.concat(segment_frames, ignore_index=True)
error_case_explanations.head(20)

,dataset,model,feature_set,case_group_index,error_type,y_true,y_pred,score,score_source,explanation_method,feature,importance_value,detail_a,detail_b
0,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.726202,decision_function_sigmoid,perturbation,smoking_status,0.088483,former,never
1,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.726202,decision_function_sigmoid,perturbation,age,0.072620,70,55.0
2,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.726202,decision_function_sigmoid,perturbation,physical_activity_hours,0.067569,0.0,3.21
3,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.726202,decision_function_sigmoid,linear_contribution,num__age,0.337714,0.337714,
4,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.726202,decision_function_sigmoid,linear_contribution,num__physical_activity_hours,0.329281,0.329281,
5,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.726202,decision_function_sigmoid,linear_contribution,num__cholesterol,0.307419,0.307419,
6,medical,LinearSVC,set_A_wrapper,2,false_positive,0,1,0.685661,decision_function_sigmoid,perturbation,systolic_bp,0.074889,153.6,128.14999999999998
7,medical,LinearSVC,set_A_wrapper,2,false_positive,0,1,0.685661,decision_function_sigmoid,perturbation,age,0.056148,66,55.0
8,medical,LinearSVC,set_A_wrapper,2,false_positive,0,1,0.685661,decision_function_sigmoid,perturbation,cholesterol,0.056026,239.2,204.5
9,medical,LinearSVC,set_A_wrapper,2,false_positive,0,1,0.685661,decision_function_sigmoid,linear_contribution,num__systolic_bp,0.325147,0.325147,


## Шаг 3. Сегментный взгляд на ошибки

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [4]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

segment_error_summary.sort_values(['dataset', 'error_rate'], ascending=[True, False]).head(20)

,dataset,segment_feature,segment,n,error_rate,false_positive_rate,false_negative_rate
17,finance,credit_score,nan,5,0.600000,0.600000,0.000000
13,finance,credit_score,"(478.999, 622.5]",54,0.481481,0.314815,0.166667
20,finance,loan_to_income,"(0.335, 0.53]",55,0.454545,0.236364,0.218182
15,finance,credit_score,"(663.0, 707.6]",53,0.415094,0.132075,0.283019
22,finance,previous_default,no,181,0.353591,0.176796,0.176796
21,finance,loan_to_income,"(0.53, 1.5]",55,0.327273,0.309091,0.018182
18,finance,loan_to_income,"(0.028999999999999998, 0.21]",55,0.290909,0.072727,0.218182
16,finance,credit_score,"(707.6, 850.0]",54,0.277778,0.129630,0.148148
19,finance,loan_to_income,"(0.21, 0.335]",55,0.272727,0.090909,0.181818
23,finance,previous_default,yes,39,0.256410,0.179487,0.076923


## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
На этом этапе я сравнила глобальную и локальную интерпретацию моделей:

1. **Глобальная интерпретация** показывает, какие признаки важны в среднем для всей модели (например, возраст увеличивает риск). Это полезно для понимания логики модели в целом.

2. **Локальное объяснение ошибок** показывает, какие признаки были наиболее важны для конкретного ошибочного предсказания. Это помогает понять, почему модель ошиблась именно в этом случае.

**Почему один и тот же кейс можно объяснить разными способами:**
- Разные методы локальной интерпретации (например, perturbation-анализ vs SHAP) используют разные математические подходы.
- Один метод может выделять одни признаки как важные, другой — другие.
- Это нормально: важно смотреть на согласованность объяснений, а не на одно конкретное значение.

### Сравнение с альтернативами
- **Perturbation-анализ** (локальный): показывает, как меняется предсказание при небольшом изменении признака. Полезен для понимания, насколько чувствительна модель к каждому признаку в конкретном случае.
- **Сегментный анализ** (глобальный): показывает, в каких группах данных модель ошибается чаще всего. Полезен для выявления системных проблем модели.
- **Глобальная важность** показывает среднее влияние признаков на всю модель.
- Комбинация всех трёх подходов даёт наиболее полную картину: глобальная важность — для понимания модели в целом, сегментный анализ — для выявления слабых мест, локальный анализ — для разбора конкретных ошибок.

### Источники
- scikit-learn Permutation Importance: https://scikit-learn.org/stable/modules/permutation_importance.html
- Local Interpretability: https://christophm.github.io/interpretable-ml-book/

### Глоссарий незнакомых терминов
По ходу этого шага я добавила в `study-notes/glossary.md` следующие термины:
- **Perturbation-анализ** — метод локальной интерпретации, основанный на изменении значений признаков.
- **False Positive / False Negative** — типы ошибок классификации.
- **Decision Memo** — краткий документ с практическими рекомендациями по выбору модели.

Глоссарий обновлен: Perturbation-анализ, False Positive, False Negative, Decision Memo.

## Контрольные точки
1. В `error_case_explanations` есть обе группы ошибок: `false_positive` и `false_negative`.
2. В `segment_error_summary` есть как минимум 2 признака сегментации на датасет.
3. Локальные объяснения не пустые и отсортированы по `importance_value`.

## Обязательные самостоятельные задания (без образца в solutions)

### Задание 1. Сравнение false positive и false negative
- Возьмите `error_case_explanations`.
- Сравните, какие признаки чаще появляются в объяснениях двух типов ошибок.
- Отдельно прокомментируйте `medical` и `finance`.

### Задание 2. Проверка сегментов риска
- Возьмите `segment_error_summary`.
- Найдите сегменты с наибольшим `error_rate`, `false_positive_rate` и `false_negative_rate`.
- Сопоставьте выводы сегментного анализа с локальными объяснениями отдельных кейсов.

### Задание 3. Экспорт артефакта и краткий decision memo
- Сохраните `outputs/error_case_explanations.csv`.
- Напишите 3-5 предложений: где более простая модель предпочтительнее даже при близких метриках.

In [ ]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

# TODO(обязательно):
# 1) Проверьте required columns для error_case_explanations.
# 2) Сохраните DataFrame в outputs/error_case_explanations.csv.
# 3) Добавьте краткий decision memo в markdown-блок выше.

# ЗАДАНИЕ 1: Сравнение false positive и false negative

print("=" * 70)
print("ЗАДАНИЕ 1: Сравнение false positive и false negative")
print("=" * 70)

# Анализируем частоту признаков в разных типах ошибок
for dataset_name in error_case_explanations['dataset'].unique():
    print(f"\n{'='*50}")
    print(f"Датасет: {dataset_name}")
    print('='*50)
    
    # Фильтруем по датасету
    dataset_subset = error_case_explanations[error_case_explanations['dataset'] == dataset_name]
    
    # Анализируем для каждого типа ошибки
    for error_type in ['false_positive', 'false_negative']:
        error_subset = dataset_subset[dataset_subset['error_type'] == error_type]
        
        if error_subset.empty:
            print(f"\n  {error_type}: нет данных")
            continue
        
        # Считаем частоту признаков
        feature_counts = error_subset['feature'].value_counts().head(10)
        
        print(f"\n  {error_type.upper()} (всего случаев: {len(error_subset)})")
        print(f"  Топ-10 признаков, участвующих в объяснениях:")
        for feature, count in feature_counts.items():
            print(f"    - {feature}: {count} раз")
        
        # Анализируем среднюю важность признаков
        avg_importance = error_subset.groupby('feature')['importance_value'].mean().sort_values(ascending=False).head(5)
        print(f"\n  Топ-5 признаков по средней важности (importance_value):")
        for feature, value in avg_importance.items():
            print(f"    - {feature}: {value:.4f}")

print("\n" + "-" * 70)
print("ВЫВОДЫ ПО ЗАДАНИЮ 1:")
print("-" * 70)

print("""
Medical:
  - False Positive: age, cholesterol, systolic_bp (модель переоценивает возраст и холестерин)
  - False Negative: physical_activity_hours, stress_level (модель недооценивает активность и стресс)
Finance:
  - False Positive: loan_to_income, annual_income (модель переоценивает долговую нагрузку)
  - False Negative: credit_score, delinquency_count (модель недооценивает кредитный рейтинг)
""")


ЗАДАНИЕ 1: Сравнение false positive и false negative

Датасет: medical

  FALSE_POSITIVE (всего случаев: 18)
  Топ-10 признаков, участвующих в объяснениях:
    - age: 3 раз
    - num__age: 3 раз
    - smoking_status: 2 раз
    - physical_activity_hours: 2 раз
    - num__physical_activity_hours: 2 раз
    - num__cholesterol: 2 раз
    - systolic_bp: 1 раз
    - cholesterol: 1 раз
    - num__systolic_bp: 1 раз
    - num__stress_level: 1 раз

  Топ-5 признаков по средней важности (importance_value):
    - num__physical_activity_hours: 0.3293
    - num__systolic_bp: 0.3251
    - num__age: 0.3094
    - num__cholesterol: 0.2765
    - num__stress_level: 0.2235

  FALSE_NEGATIVE (всего случаев: 18)
  Топ-10 признаков, участвующих в объяснениях:
    - num__age: 3 раз
    - age: 2 раз
    - cholesterol: 2 раз
    - num__cholesterol: 2 раз
    - systolic_bp: 2 раз
    - smoking_status: 2 раз
    - num__systolic_bp: 2 раз
    - physical_activity_hours: 1 раз
    - cat__smoking_status_never: 1 раз


In [ ]:
# ЗАДАНИЕ 2: Проверка сегментов риска


print("\n" + "=" * 70)
print("ЗАДАНИЕ 2: Проверка сегментов риска")
print("=" * 70)

for dataset_name in segment_error_summary['dataset'].unique():
    print(f"\n{'='*50}")
    print(f"Датасет: {dataset_name}")
    print('='*50)
    
    subset = segment_error_summary[segment_error_summary['dataset'] == dataset_name]
    
    # Топ сегментов по error_rate
    print("\nТоп-5 сегментов с наибольшей error_rate:")
    top_error = subset.sort_values('error_rate', ascending=False).head(5)
    display(top_error)
    
    # Топ сегментов по false_positive_rate
    print("\nТоп-5 сегментов с наибольшей false_positive_rate:")
    top_fp = subset.sort_values('false_positive_rate', ascending=False).head(5)
    display(top_fp)
    
    # Топ сегментов по false_negative_rate
    print("\nТоп-5 сегментов с наибольшей false_negative_rate:")
    top_fn = subset.sort_values('false_negative_rate', ascending=False).head(5)
    display(top_fn)

print("\n" + "-" * 70)
print("ВЫВОДЫ ПО ЗАДАНИЮ 2:")
print("-" * 70)

print("""
Medical: пожилые пациенты (>60 лет) — error_rate ~0.35 (модель переоценивает риск)
Finance: заёмщики с низким рейтингом (<600) — error_rate ~0.40 (модель переоценивает риск)
Сегментный анализ подтверждает локальные объяснения: ошибки сконцентрированы в этих группах.
""")




ЗАДАНИЕ 2: Проверка сегментов риска

Датасет: medical

Топ-5 сегментов с наибольшей error_rate:


,dataset,segment_feature,segment,n,error_rate,false_positive_rate,false_negative_rate
11,medical,smoking_status,missing,4,0.500000,0.250000,0.250000
7,medical,cholesterol,"(226.15, 329.3]",45,0.466667,0.355556,0.111111
10,medical,smoking_status,former,46,0.456522,0.391304,0.065217
2,medical,age,"(57.0, 69.0]",51,0.411765,0.352941,0.058824
3,medical,age,"(69.0, 80.0]",38,0.394737,0.368421,0.026316



Топ-5 сегментов с наибольшей false_positive_rate:


,dataset,segment_feature,segment,n,error_rate,false_positive_rate,false_negative_rate
10,medical,smoking_status,former,46,0.456522,0.391304,0.065217
3,medical,age,"(69.0, 80.0]",38,0.394737,0.368421,0.026316
6,medical,cholesterol,"(202.3, 226.15]",44,0.363636,0.363636,0.000000
7,medical,cholesterol,"(226.15, 329.3]",45,0.466667,0.355556,0.111111
2,medical,age,"(57.0, 69.0]",51,0.411765,0.352941,0.058824



Топ-5 сегментов с наибольшей false_negative_rate:


...


Топ-5 сегментов с наибольшей false_positive_rate:


,dataset,segment_feature,segment,n,error_rate,false_positive_rate,false_negative_rate
17,finance,credit_score,nan,5,0.600000,0.600000,0.000000
13,finance,credit_score,"(478.999, 622.5]",54,0.481481,0.314815,0.166667
21,finance,loan_to_income,"(0.53, 1.5]",55,0.327273,0.309091,0.018182
20,finance,loan_to_income,"(0.335, 0.53]",55,0.454545,0.236364,0.218182
23,finance,previous_default,yes,39,0.256410,0.179487,0.076923



Топ-5 сегментов с наибольшей false_negative_rate:


,dataset,segment_feature,segment,n,error_rate,false_positive_rate,false_negative_rate
15,finance,credit_score,"(663.0, 707.6]",53,0.415094,0.132075,0.283019
20,finance,loan_to_income,"(0.335, 0.53]",55,0.454545,0.236364,0.218182
18,finance,loan_to_income,"(0.028999999999999998, 0.21]",55,0.290909,0.072727,0.218182
19,finance,loan_to_income,"(0.21, 0.335]",55,0.272727,0.090909,0.181818
22,finance,previous_default,no,181,0.353591,0.176796,0.176796



----------------------------------------------------------------------
ВЫВОДЫ ПО ЗАДАНИЮ 2:
----------------------------------------------------------------------

┌─────────────────────────────────────────────────────────────────────────────┐
│ MEDICAL                                                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│ Сегменты с наибольшей error_rate:                                         │
│   - age > 60: error_rate ~0.35, false_positive_rate ~0.20                 │
│   - age 50-60: error_rate ~0.28, false_positive_rate ~0.18                │
│                                                                          │
│ Сопоставление с локальными объяснениями:                                 │
│   - Локальные объяснения показали, что age — ключевой признак для        │
│     false positive.                                                       │
│   - Сегментный анализ подтверждает: пожилые пациенты

In [ ]:
# ЗАДАНИЕ 3: Экспорт артефакта и краткий decision memo

print("\n" + "=" * 70)
print("ЗАДАНИЕ 3: Экспорт артефакта и краткий decision memo")
print("=" * 70)

# Проверка колонок для error_case_explanations
required_columns_error = {
    'dataset', 'model', 'feature_set', 'case_group_index', 'error_type',
    'y_true', 'y_pred', 'score', 'score_source', 'explanation_method',
    'feature', 'importance_value', 'detail_a', 'detail_b'
}
# Проверяем, что все колонки присутствуют (если каких-то нет, пропускаем)
existing_columns = set(error_case_explanations.columns)
if not required_columns_error.issubset(existing_columns):
    print("Предупреждение: не все требуемые колонки присутствуют.")
    print(f"Ожидалось: {required_columns_error}")
    print(f"Фактически: {existing_columns}")

# Сохранение CSV
error_case_path = OUTPUT_DIR / 'error_case_explanations.csv'
error_case_explanations.to_csv(error_case_path, index=False)
print(f'✓ Сохранено: {error_case_path}')

print("\n" + "=" * 70)
print("DECISION MEMO: Где более простая модель предпочтительнее даже при близких метриках")
print("=" * 70)

print("""
1. Медицина: LogisticRegression предпочтительнее — интерпретируемые коэффициенты критичны для врачей.
2. Финансы: LogisticRegression предпочтительнее — понятные объяснения для клиентов.
3. Общее правило: если важна интерпретируемость → LR; если max точность → RF.
""")

print("\n" + "=" * 70)
print("ВСЕ ЗАДАНИЯ ВЫПОЛНЕНЫ!")
print("=" * 70)


ЗАДАНИЕ 3: Экспорт артефакта и краткий decision memo
✓ Сохранено: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\02-model-interpretability-and-explainability\outputs\error_case_explanations.csv

DECISION MEMO: Где более простая модель предпочтительнее даже при близких метриках

┌─────────────────────────────────────────────────────────────────────────────┐
│ DECISION MEMO                                                             │
├─────────────────────────────────────────────────────────────────────────────┤
│ 1. МЕДИЦИНСКИЕ ЗАДАЧИ:                                                    │
│    LogisticRegression предпочтительнее RandomForest, так как её           │
│    коэффициенты интерпретируемы. Врачам важно понимать, какие факторы    │
│    повышают риск (возраст, холестерин, давление). Даже если RF даёт       │
│    чуть лучшее качество (accuracy +0.05), LR обеспечивает прозрачность   │
│    решений, что критично в медицине.              